<a href="https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/dealing_with_field_point_data/lammermuir_combining_river_and_point_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/dealing_with_field_point_data/lammermuir_combining_river_and_point_data.ipynb)

# Combining field measurements with a river network: the Kilmade Burn, Lammermuir Hills

Measurements made in the field, such as channel widths or grain sizes, are made **next to** a river. To compare them with properties of the river network (drainage area, channel gradient, chi), each measurement has to be tied to a channel cell of the DEM. A handheld GPS is good to a few metres; a channel on a 1 m DEM is a line of cells one metre wide. So the recorded positions almost never fall on the channel.

This notebook works through that problem for the **Kilmade Burn**, a small tributary of the Whiteadder Water in the Lammermuir Hills, south-east Scotland. We:

1. Download **1 m lidar** (Scottish Remote Sensing Portal, phase 3) with `lsdtt-fetch-raster`.
2. Compute the **drainage area**, the **chi coordinate** and the **channel gradient** of the burn's channel network.
3. Make a set of **simulated field measurements**: channel width `w` and the grain sizes `d50` and `d84` at about 30 sites, with GPS positions a few metres off the channel.
4. **Snap** the field points to the channel network with `lsdtt-channel-extraction` (`run_channel_point_attributes`), which adds drainage area and channel gradient to each point, and join chi to them.
5. Plot width against drainage area, and map the points.
6. Compare the channel gradient with the gradient of a **polynomial surface** fitted to the DEM.

The field data here are made up, so that you can see what is going on. With your own field data you would skip Step 4 and read your own CSV file.

Each `lsdtt3` program is controlled by a small **parameter file**: a text file of `key: value` lines. We write these files from Python and then run the programs with `!`.

## Setup

### Get the lsdtt3 command-line tools

The Linux package of the latest [lsdtt3 release](https://github.com/LSDtopotools/lsdtt3/releases/latest). `releases/latest/download/` always points at the newest version.

In [ ]:
!wget -q https://github.com/LSDtopotools/lsdtt3/releases/latest/download/lsdtt3-backend-linux-x86_64-core.tar.gz
!tar -xzf lsdtt3-backend-linux-x86_64-core.tar.gz

Now we tell the system where to find the `lsdtt3` programs and the libraries they need. If your runtime restarts later, run this cell again.

In [ ]:
import glob
import os

root = sorted(glob.glob("/content/lsdtt3-backend-linux-x86_64-core-v*"))[-1]
print("Using", root)
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's proj.db breaks rasterio and pyproj, which we use for the plots.

In [ ]:
!lsdtt-channel-extraction -v

We read the rasters with `rasterio`, and the points with `geopandas` (already in Colab).

In [ ]:
!pip install -q rasterio

## Step 1: 1 m lidar of the Kilmade Burn

The Scottish Government's lidar programme covers much of Scotland. `lsdtt-fetch-raster` gets it from the [Scottish Remote Sensing Portal](https://remotesensingdata.gov.scot/) with no account or key (`dem_source: gb_sct_lidar`). **Phase 3** covers the Lammermuirs with a 50 cm DTM; we ask for **1 m** (`grid_spacing: 1`), which is plenty for a burn a metre or two wide, and four times smaller than the 50cm DEM.

The DEM comes in British National Grid (EPSG:27700), so all positions in this notebook are **eastings and northings in metres**.

### Why the box is bigger than the burn

A drainage area is only right if the DEM contains the **whole catchment** upstream. The box first drawn around the burn, from (55.8388, -2.5626) to (55.8631, -2.5238), cuts off its headwaters: they extend about 250 m south of it. `lsdtt3` marks a catchment that touches the edge of the DEM as **incomplete**, and the chi analysis leaves it out. So we move the south and west edges out until the whole catchment fits, with a few hundred metres to spare.

The download is about 10 million cells and takes around half a minute.

In [ ]:
# Lower left (south-west) and upper right (north-east) corners, as copied from Google Maps
lower_left = (55.832, -2.567)
upper_right = (55.86307193400781, -2.523841493895955)

with open("kilmade_fetch.param", "w") as f:
    f.write("write_fname: kilmade\n")
    f.write(f"west: {lower_left[1]}\n")
    f.write(f"east: {upper_right[1]}\n")
    f.write(f"south: {lower_left[0]}\n")
    f.write(f"north: {upper_right[0]}\n")
    f.write("dem_source: gb_sct_lidar\n")
    f.write("source_collection_choice: phase-3\n")
    f.write("grid_spacing: 1\n")
    f.write("resampling_method: average\n")
    f.write("target_epsg: 27700\n")
!lsdtt-fetch-raster ./ kilmade_fetch.param

We also make a hillshade, for the maps, which we will use later:

In [ ]:
with open("kilmade_hillshade.param", "w") as f:
    f.write("read_fname: kilmade_DEM.tif\n")
    f.write("write_fname: kilmade_hs\n")
    f.write("write_output_format: gtiff\n")
    f.write("write_hillshade_raster: true\n")
!lsdtt-raster-preprocessing ./ kilmade_hillshade.param

## Step 2: Drainage area and chi along the burn

**The outlet.** The Kilmade Burn joins the Whiteadder Water in the north-east of the DEM. We put the outlet a few metres up the burn from the confluence, at easting 366640.5, northing 663208.5, so that the catchment is the burn's alone and not the Whiteadder's 50 km². `lsdtt3` reads outlet points from a CSV with `easting` and `northing` columns in the DEM's coordinate system (or `latitude` and `longitude`).

**The channels.** A channel cell is one draining at least `threshold_contributing_pixels` cells: 100000 here, which on a 1 m DEM is 100000 m², or 0.1 km². This is a simple threshold: the channels start where the drainage area reaches it, which is not exactly where the real channel heads are.

**Chi.** The chi coordinate (Perron and Royden, 2013) is an integral of drainage area along the channel, from the outlet upstream:

```
χ = ∫ (A₀ / A(x))^(m/n) dx
```

A channel in steady state has elevation proportional to χ. We use `m_over_n: 0.45`, a common default.

`write_chi_data_maps` writes one row per channel cell: its position, chi, drainage area (m²), flow distance (m), elevation, and a `source_key` that identifies each channel from its source down to where it joins a bigger one.

In [ ]:
import pandas as pd

pd.DataFrame({"easting": [366640.5], "northing": [663208.5]}).to_csv("kilmade_outlet.csv", index=False)

with open("kilmade_chi.param", "w") as f:
    f.write("read_fname: kilmade_DEM.tif\n")
    f.write("write_fname: kilmade\n")
    f.write("threshold_contributing_pixels:  100000\n")
    f.write("basin_outlet_fname: kilmade_outlet.csv\n")
    f.write("basin_outlet_snapping_method: downstream\n")
    f.write("basin_outlet_contributing_pixels_threshold:  100000\n")
    f.write("m_over_n: 0.45\n")
    f.write("write_chi_data_maps: true\n")
    f.write("chi_data_maps_write_vector_format: csv\n")
!lsdtt-chi-analysis ./ kilmade_chi.param

The chi data are in latitude and longitude. We convert them to the DEM's British National Grid, and list the biggest channels: the **main stem** and its tributaries, with their drainage area where they join.

In [ ]:
import geopandas as gpd
import numpy as np

chi = pd.read_csv("kilmade_chi_data_maps.csv")
chi = gpd.GeoDataFrame(chi, geometry=gpd.points_from_xy(chi.longitude, chi.latitude), crs="EPSG:4326").to_crs("EPSG:27700")
chi["easting"], chi["northing"] = chi.geometry.x, chi.geometry.y

channels = (chi.groupby("source_key")
               .agg(n_cells=("chi", "size"), area_at_mouth_km2=("drainage_area", "max"))
               .sort_values("area_at_mouth_km2", ascending=False))
channels["area_at_mouth_km2"] /= 1e6
print(f"{chi.source_key.nunique()} channels, {len(chi)} channel cells, catchment {chi.drainage_area.max() / 1e6:.2f} km2")
print(channels.head(6).round(3))

main_stem = channels.index[0]
tributaries = list(channels.index[1:4])
print("Main stem: source_key", main_stem, "  three biggest tributaries:", tributaries)

## Step 3: The channel gradient

A gradient taken between two neighbouring cells of a 1 m DEM is mostly noise: a stone or a pool changes the elevation by more than the drop over a metre. `lsdtt-channel-extraction` instead fits a straight line to elevation against distance **along the channel**, over a window centred on each cell. `slope_window_half_length_metres: 10` makes the window 20 m long: 10 m upstream and 10 m downstream. Upstream of a junction the window follows the bigger tributary.

`write_channel_slope_points` writes this gradient for every channel cell of the DEM. The same run also snaps the field points (Step 5), so we write the parameter file once, as a function.

In [ ]:
def write_channel_param(fname, points_fname=None):
    with open(fname, "w") as f:
        f.write("read_fname: kilmade_DEM.tif\n")
        f.write("write_fname: kilmade\n")
        f.write("threshold_contributing_pixels: 5000\n")
        f.write("slope_window_half_length_metres: 10\n")
        f.write("write_vector_format: flatgeobuf\n")
        if points_fname is None:
            f.write("write_channel_slope_points: true\n")
        else:
            f.write("run_channel_point_attributes: true\n")
            f.write(f"channel_points_fname: {points_fname}\n")
            f.write("channel_points_snap_max_search_radius_pixels: 15\n")


write_channel_param("kilmade_slopes.param")
!lsdtt-channel-extraction ./ kilmade_slopes.param

In [ ]:
slopes = gpd.read_file("kilmade_channel_slope_points.fgb")
print(slopes.columns.tolist())
# Keep the cells of the Kilmade Burn: those that are in the chi network
# (np.floor of a coordinate is the cell it falls in: cell centres are at .5 m)
cell_id = lambda x, y: np.floor(x).astype(np.int64) * 10**7 + np.floor(y).astype(np.int64)
in_burn = np.isin(cell_id(slopes.geometry.x, slopes.geometry.y), cell_id(chi.easting, chi.northing))
slopes = slopes[in_burn]
print(f"{len(slopes)} channel cells in the Kilmade Burn; median gradient {slopes.slope_m_per_m.median():.3f}")

## Step 4: Simulated field data

**WE ARE MAKING UP SOME DATA HERE.** If you have your own data I'll tell you later where to add that.

Now we invent a field campaign: **30 sites**, 20 along the main stem and 10 on the three biggest tributaries. At each site a field team has recorded

* `w`: the channel width, in metres;
* `d50` and `d84`: the grain sizes that 50% and 84% of the bed is finer than, in metres;
* `easting` and `northing`: the position from a handheld GPS.

We build the "true" values from simple rules, then add noise as a field measurement would have:

* **Width** follows the hydraulic-geometry rule `w = k A^0.5`, with `k` chosen so that the burn is about 1.5 m wide where it drains 1 km², and 20% scatter.
* **Grain size** coarsens gently downstream here (d50 about 8 cm, d84 about 15 cm), with scatter. Grain size can go either way in real rivers: fining downstream by abrasion and sorting is common, and coarsening, as here, happens where steep tributaries or valley sides supply coarse sediment.
* **The GPS position** is the true channel position moved **3 to 10 m** in a random direction.

We skip the first 60 m of the main stem, so that no site is close enough to be snapped onto the Whiteadder Water instead of the burn. We also keep the main-stem sites below its dead-straight top section along the southern divide (where it drains less than about 0.06 km²): a channel that straight is probably man-made, a ditch or a track, and not where a field team would measure the burn. `np.random.default_rng(42)` makes the same "random" field data every time.

In [ ]:
rng = np.random.default_rng(42)


def pick_sites(source_key, n, skip_m=0, min_area_m2=20000):
    """n channel cells of one channel, evenly spaced along it, at least skip_m from its
    mouth and draining at least min_area_m2."""
    c = chi[chi.source_key == source_key].sort_values("flow_distance")
    c = c[c.flow_distance >= c.flow_distance.min() + skip_m]
    c = c[c.drainage_area >= min_area_m2]
    targets = np.linspace(c.flow_distance.min(), c.flow_distance.max(), n)
    return c.iloc[[np.abs(c.flow_distance.to_numpy() - t).argmin() for t in targets]]


sites = pd.concat([pick_sites(main_stem, 20, skip_m=60, min_area_m2=60000)] +
                  [pick_sites(k, n) for k, n in zip(tributaries, [4, 3, 3])], ignore_index=True)

area_km2 = sites.drainage_area.to_numpy() / 1e6
n = len(sites)
w = 1.5 * area_km2 ** 0.5 * rng.lognormal(0, 0.2, n)
d50 = 0.08 * area_km2 ** 0.15 * rng.lognormal(0, 0.25, n)
d84 = d50 * 1.9 * rng.lognormal(0, 0.1, n)

offset = rng.uniform(3, 10, n)
direction = rng.uniform(0, 2 * np.pi, n)
field = pd.DataFrame({"site": [f"K{i + 1:02d}" for i in range(n)],
                      "easting": (sites.easting + offset * np.cos(direction)).round(1),
                      "northing": (sites.northing + offset * np.sin(direction)).round(1),
                      "w": w.round(2), "d50": d50.round(3), "d84": d84.round(3)})
field.to_csv("kilmade_field_data.csv", index=False)
print(field.head(8).to_string(index=False))
print(f"... {n} sites. Median w {np.median(w):.2f} m, d50 {np.median(d50):.3f} m, d84 {np.median(d84):.3f} m")

`kilmade_field_data.csv` is what you would bring back from the field: a site name, the GPS position and the measurements. **Nothing in it says which channel cell a site belongs to.** From here on we use only this file, as you would with real data.

## Step 5: Snapping the field points to the channel

**IF YOU HAVE YOUR OWN DATA CHANGE THE `points_fname` IN THE CELL BELOW.**

`run_channel_point_attributes` moves each point to the **nearest channel cell**, in a straight line, searching up to `channel_points_snap_max_search_radius_pixels` cells away (15 cells is 15 m here). It does not follow the point's flow path downhill: a point on a bank would otherwise drain down the hillside and could join the channel far downstream.

It keeps every column of the input and adds, among others, `snap_distance_m`, `drainage_area_m2`, `flow_distance_m`, `elevation_m` and `slope_m_per_m`: the 20 m along-channel gradient of Step 3. A point with no channel within the radius is kept where it was, with `snap_status` saying why and the added values left empty.

The points come in as a CSV with `easting` and `northing` in the DEM's coordinate system. We write the output as FlatGeobuf (`.fgb`), a vector format QGIS and geopandas read directly.

In [ ]:
write_channel_param("kilmade_snap.param", points_fname="kilmade_field_data.csv")
print("This is the driver file for the point snapping:")
!cat kilmade_snap.param

In [ ]:

!lsdtt-channel-extraction ./ kilmade_snap.param

In [ ]:
snapped = gpd.read_file("kilmade_channel_points.fgb")
print(snapped.snap_status.value_counts().to_string())
print(f"Snap distance: median {snapped.snap_distance_m.median():.1f} m, largest {snapped.snap_distance_m.max():.1f} m")

### Adding chi

`run_channel_point_attributes` does not compute chi, but the snapped points now sit **exactly on channel cells**, the same cells the chi data maps of Step 2 list. So we can look chi up by position: for each snapped point, the chi data map cell at the same place. We check that every point finds a cell less than a metre away, which means the same cell.

In [ ]:
joined = gpd.sjoin_nearest(snapped, chi[["chi", "source_key", "geometry"]], how="left",
                           max_distance=1.0, distance_col="chi_match_distance_m")
joined = joined.drop(columns="index_right").sort_values("site").reset_index(drop=True)
print(f"{joined.chi.notna().sum()} of {len(joined)} points have chi")
print(joined[["site", "w", "d50", "d84", "snap_distance_m", "drainage_area_m2", "slope_m_per_m", "chi"]]
      .head(8).round(4).to_string(index=False))

joined.to_file("kilmade_field_data_snapped.fgb", driver="FlatGeobuf")
out = joined.drop(columns="geometry")
out.insert(1, "snapped_easting", joined.geometry.x)
out.insert(2, "snapped_northing", joined.geometry.y)
out.to_csv("kilmade_field_data_snapped.csv", index=False)

### How good is the snapping?

With real data we would not know where each site truly was. Here we do: we made the data from the channel cells in `sites`. So we can measure how far each snapped point is from its **true** cell, and whether it landed on the right channel.

The snap distance (how far the point moved) is not the error. A GPS error **along** the channel moves the point to another cell of the same channel, a few metres up- or downstream. It snaps back a short distance, but onto the wrong cell. An error **across** the channel is undone by the snapping.

In [ ]:
truth = sites.assign(site=[f"K{i + 1:02d}" for i in range(len(sites))]).set_index("site")
check = joined.set_index("site")
error_m = np.hypot(check.geometry.x - truth.easting, check.geometry.y - truth.northing)
gps_error_m = np.hypot(check.original_x - truth.easting, check.original_y - truth.northing)
print(f"GPS error:                 median {gps_error_m.median():.1f} m, largest {gps_error_m.max():.1f} m")
print(f"Snapped point to true cell: median {error_m.median():.1f} m, largest {error_m.max():.1f} m")
print(f"Points on the wrong channel: {(check.source_key != truth.source_key).sum()} of {len(check)}")
wrong = check.source_key != truth.source_key
print(pd.DataFrame({"true area (km2)": truth.drainage_area[wrong] / 1e6,
                    "snapped area (km2)": check.drainage_area_m2[wrong] / 1e6,
                    "GPS error (m)": gps_error_m[wrong]}).round(3))

**Confluences are the trap.** The points on the wrong channel are sites at the **mouth of a tributary**: the first site of each tributary is placed right where it joins the main stem (or a bigger branch), and a GPS error of a few metres put it closer to the bigger channel. It then takes the bigger channel's drainage area: several times too large. Such a point stands out in the width plot below, a narrow tributary width at a main-stem drainage area.

With real data, note in the field which channel you are on. Then check that each point snapped onto it, and move the ones that did not before you snap them.

`kilmade_field_data_snapped.csv` and `kilmade_field_data_snapped.fgb` now hold every field measurement with the channel properties of its cell. In the CSV, `easting`/`northing` are the GPS positions and `snapped_easting`/`snapped_northing` the channel cells.

**Check the snapping.** A point recorded near a confluence may be closer to the other channel. Points that moved much further than your GPS error, or that changed channel (`source_key` different from where you think you were), are worth a second look.

## Step 6: Width against drainage area

Hydraulic geometry says width grows as a power of discharge, and so, roughly, of drainage area: `w = k A^b`. On log axes a power law is a straight line, of slope `b`. We fit it by least squares to log w against log A.

In [ ]:
import matplotlib.pyplot as plt

ok = joined.drainage_area_m2.notna()
A_km2 = joined.drainage_area_m2[ok] / 1e6
b, log_k = np.polyfit(np.log10(A_km2), np.log10(joined.w[ok]), 1)
print(f"Fit: w = {10 ** log_k:.2f} A^{b:.2f}   (w in m, A in km2)")

on_main = joined.source_key[ok] == main_stem
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(A_km2[on_main], joined.w[ok][on_main], color="tab:blue", label="main stem")
ax.scatter(A_km2[~on_main], joined.w[ok][~on_main], color="tab:orange", label="tributaries")
A_line = np.logspace(np.log10(A_km2.min()), np.log10(A_km2.max()), 50)
ax.plot(A_line, 10 ** log_k * A_line ** b, "k--", label=f"w = {10 ** log_k:.2f} A$^{{{b:.2f}}}$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Drainage area (km²)")
ax.set_ylabel("Channel width, w (m)")
ax.set_yticks([0.2, 0.5, 1, 2, 3])
ax.yaxis.set_major_formatter(plt.FormatStrFormatter("%g"))
ax.yaxis.set_minor_formatter(plt.NullFormatter())
ax.xaxis.set_major_formatter(plt.FormatStrFormatter("%g"))
ax.legend()
plt.show()

The data were made with `b = 0.5` and `k = 1.5`, so the fit should come close to those. With only 30 noisy points it will not be exact: try another seed in Step 4 and see how much the fitted exponent changes. That spread is a fair guide to how well a field campaign of this size pins down `b`.

## Step 7: Map of the field points

The hillshade, the catchment of the Kilmade Burn (white) and its channels (blue), the field points where the GPS put them (open circles), the channel cells they were snapped to (filled), and a line between each pair.

In [ ]:
import rasterio
from matplotlib.collections import LineCollection


def read_raster(fname):
    with rasterio.open(fname) as src:
        data = src.read(1).astype(float)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan
        left, bottom, right, top = src.bounds
    return data, [left, right, bottom, top]


hillshade, extent = read_raster("kilmade_hs_hillshade.tif")
catchment = gpd.read_file("kilmade_chi_basin_polygons.fgb")
xmin, ymin, xmax, ymax = catchment.total_bounds + np.array([-100, -100, 100, 100])


def base_map(ax, channel_color="tab:blue", channel_size=0.3):
    ax.imshow(hillshade, cmap="gray", extent=extent)
    catchment.boundary.plot(ax=ax, color="white", lw=1)
    ax.scatter(chi.easting, chi.northing, s=channel_size, color=channel_color, label="channels")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")


fig, ax = plt.subplots(figsize=(9, 11))
base_map(ax)
ax.add_collection(LineCollection([[(r.original_x, r.original_y), (r.geometry.x, r.geometry.y)]
                                  for r in joined.itertuples()], colors="yellow", lw=1))
ax.scatter(joined.original_x, joined.original_y, s=40, facecolors="none", edgecolors="red", label="GPS position")
ax.scatter(joined.geometry.x, joined.geometry.y, s=15, color="yellow", edgecolors="black", lw=0.5, label="snapped")
ax.legend(loc="lower left")
plt.show()

At this scale the snapping lines are only a few pixels long. Zoom in on one site to see it:

In [ ]:
site = joined.iloc[10]
fig, ax = plt.subplots(figsize=(6, 6))
base_map(ax)
ax.plot([site.original_x, site.geometry.x], [site.original_y, site.geometry.y], color="yellow")
ax.scatter(site.original_x, site.original_y, s=80, facecolors="none", edgecolors="red", label="GPS position")
ax.scatter(site.geometry.x, site.geometry.y, s=40, color="yellow", edgecolors="black", label="snapped")
ax.scatter(chi.easting, chi.northing, s=8, color="tab:blue")
ax.set_xlim(site.geometry.x - 30, site.geometry.x + 30)
ax.set_ylim(site.geometry.y - 30, site.geometry.y + 30)
ax.set_title(f"Site {site.site}: moved {site.snap_distance_m:.1f} m")
ax.legend()
plt.show()

## Step 8: Gradient from a fitted surface

The along-channel gradient of Step 3 follows the channel. A different, very common gradient comes from fitting a **quadratic surface** to the DEM in a small window around each cell and taking the steepest slope of that surface (`lsdtt-surface-metrics`, `polyfit_metrics: slope`). It ignores where the channel goes: at a channel cell it measures the steepest direction of the ground, which in a narrow valley can be **across** the channel, down the valley side.

We fit over `polyfit_lambda_metres: 4`, a window a few metres across.

In [ ]:
with open("kilmade_polyfit.param", "w") as f:
    f.write("read_fname: kilmade_DEM.tif\n")
    f.write("write_fname: kilmade\n")
    f.write("polyfit_metrics: slope\n")
    f.write("polyfit_lambda_metres: 4\n")
!lsdtt-surface-metrics ./ kilmade_polyfit.param

In [ ]:
polyfit_tif = glob.glob("kilmade_*polyfit_slope.tif")[0]
polyfit_slope, _ = read_raster(polyfit_tif)

# The polyfit gradient at each snapped point
with rasterio.open(polyfit_tif) as src:
    joined["polyfit_slope_4m"] = [v[0] for v in src.sample(zip(joined.geometry.x, joined.geometry.y))]

fig, ax = plt.subplots(figsize=(9, 11))
base_map(ax, channel_color="white", channel_size=0.05)
im = ax.imshow(polyfit_slope, cmap="magma_r", extent=extent, vmin=0, vmax=1, alpha=0.6)
fig.colorbar(im, ax=ax, shrink=0.6, label="Polyfit gradient, 4 m window (m/m)")
pts = ax.scatter(joined.geometry.x, joined.geometry.y, c=joined.slope_m_per_m, cmap="viridis",
                 vmin=0, vmax=joined.slope_m_per_m.quantile(0.95), s=50, edgecolors="black", zorder=3)
fig.colorbar(pts, ax=ax, shrink=0.6, label="Along-channel gradient at the snapped points, 20 m window (m/m)")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(joined.slope_m_per_m, joined.polyfit_slope_4m)
top = max(joined.slope_m_per_m.max(), joined.polyfit_slope_4m.max()) * 1.05
ax.plot([0, top], [0, top], "k--", lw=1, label="1:1")
ax.set_xlabel("Along-channel gradient, 20 m window (m/m)")
ax.set_ylabel("Polyfit gradient, 4 m window (m/m)")
ax.legend()
plt.show()
print(f"Median along-channel {joined.slope_m_per_m.median():.3f}, median polyfit {joined.polyfit_slope_4m.median():.3f}")

**Reading it.** Points well above the 1:1 line are channel cells where the fitted surface's steepest direction is down the valley side or a bank, not along the bed. The polyfit gradient answers "how steep is the ground here?"; the along-channel gradient answers "how steep is the river?". For relating channel gradient to grain size or width, use the along-channel one.

## Things to try

* **Grain size against gradient.** Plot `d50` against `slope_m_per_m`, or against the shear-stress proxy `slope_m_per_m * drainage_area_m2 ** 0.5`.
* **Another seed.** Change `default_rng(42)` and see how much the fitted width exponent moves.
* **Bigger GPS errors.** Allow offsets up to 25 m in Step 4. Which points now snap to the wrong channel? Does `channel_points_snap_max_search_radius_pixels` need to grow?
* **The slope window.** Try `slope_window_half_length_metres: 25` or `5`. A longer window is smoother but averages over steps and pools.
* **Your own data.** Replace `kilmade_field_data.csv` with your own CSV: `easting` and `northing` in British National Grid, or `latitude` and `longitude`, plus any measurement columns.

## Data and references

* Lidar: Scottish Public Sector LiDAR Phase 3, Scottish Government, from the Scottish Remote Sensing Portal (https://remotesensingdata.gov.scot/). Contains public sector information licensed under the Open Government Licence v3.0.
* Perron, J. T. and Royden, L. (2013). An integral approach to bedrock river profile analysis. *Earth Surface Processes and Landforms*, 38(6), 570–576. https://doi.org/10.1002/esp.3302
* Snapping field points: https://lsdtopotools.github.io/lsdtt3/design_and_algorithms/channel_points.html
* Every input of the programs used: https://lsdtopotools.github.io/lsdtt3/ (or run any program with `-h`).